# keyword_review — Keyword Scanner

Scan text files for one or more keywords and report all matches.

**How to use:**
1. Edit the **Configuration** cell below (keywords, file patterns, options).
2. Run all cells (`Kernel → Restart & Run All`).
3. Matches are printed inline; an optional CSV is written if `OUTPUT_FILE` is set.

In [ ]:
import collections
import csv
import glob
import sys

## Helper functions

In [ ]:
def resolve_files(patterns):
    """Expand glob patterns into a deduplicated, sorted list of file paths."""
    seen = {}
    for pattern in patterns:
        matches = glob.glob(pattern)
        if not matches:
            print(f"Warning: no files matched pattern '{pattern}'", file=sys.stderr)
        for path in matches:
            seen[path] = None
    return sorted(seen.keys())


def scan_file(path, keywords, case_sensitive):
    """
    Scan a single file for keyword matches.

    Returns a list of dicts with keys: file, line_no, keyword, line.
    Prints a warning and returns [] on any file-access or permission error.
    """
    try:
        with open(path, encoding="utf-8", errors="replace") as fh:
            lines = fh.readlines()
    except FileNotFoundError:
        print(f"Warning: file not found: '{path}'", file=sys.stderr)
        return []
    except PermissionError:
        print(f"Warning: permission denied: '{path}'", file=sys.stderr)
        return []

    matches = []
    for line_no, raw_line in enumerate(lines, start=1):
        line = raw_line.rstrip()
        haystack = line if case_sensitive else line.lower()
        for keyword in keywords:
            needle = keyword if case_sensitive else keyword.lower()
            if needle in haystack:
                matches.append(
                    {
                        "file": path,
                        "line_no": line_no,
                        "keyword": keyword,
                        "line": line,
                    }
                )
    return matches


def print_matches(matches):
    """Print each match as: filename · line_no · keyword · line."""
    for m in matches:
        print(f"{m['file']} \u00b7 {m['line_no']} \u00b7 {m['keyword']} \u00b7 {m['line']}")


def write_csv(matches, output_path):
    """Write matches to a CSV file with headers: file, line_no, keyword, line."""
    try:
        with open(output_path, "w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(
                fh, fieldnames=["file", "line_no", "keyword", "line"]
            )
            writer.writeheader()
            writer.writerows(matches)
        print(f"Results written to '{output_path}'.")
    except PermissionError:
        print(
            f"Warning: permission denied writing to '{output_path}'. CSV not saved.",
            file=sys.stderr,
        )


def print_summary(all_files, keywords, matches):
    """Print end-of-run statistics."""
    files_with_matches = {m["file"] for m in matches}
    zero_match_files = [f for f in all_files if f not in files_with_matches]
    keyword_counts = collections.Counter(m["keyword"] for m in matches)

    print("\n--- Summary ---")
    print(f"Files scanned  : {len(all_files)}")
    print("Matches per keyword:")
    for kw in keywords:
        print(f"  {kw:<20}: {keyword_counts.get(kw, 0)}")
    if zero_match_files:
        print("Files with no matches:")
        for f in zero_match_files:
            print(f"  {f}")
    else:
        print("Files with no matches: (none)")

## Configuration

Edit these variables, then run the **Run scan** cell below.

In [ ]:
# One or more keywords to search for
KEYWORDS = ["error", "warning"]

# Glob patterns for files to scan
FILE_PATTERNS = ["*.txt", "logs/*.log"]

# Set to True for case-sensitive matching
CASE_SENSITIVE = False

# Set to a filename (e.g. "results.csv") to save output, or None to skip
OUTPUT_FILE = None

## Run scan

In [ ]:
all_files = resolve_files(FILE_PATTERNS)

all_matches = []
for path in all_files:
    all_matches.extend(scan_file(path, KEYWORDS, CASE_SENSITIVE))

print_matches(all_matches)

if OUTPUT_FILE:
    write_csv(all_matches, OUTPUT_FILE)

print_summary(all_files, KEYWORDS, all_matches)